In [1]:
import json
with open("../../instruction_following/notebooks/instruction-data.json", "r") as file:
    test_data = json.load(file)

In [2]:
len(test_data)

1100

In [3]:
import requests

def query_model(prompt, model="llama3", url="http://localhost:11434/api/chat"):
    data = {
        "model": model,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "options": {
            "seed": 123, 
            "temperature": 0,
            "num_ctx": 2048
        },
    }
    with requests.post(url=url, json=data, stream=True, timeout=30) as r:
        r.raise_for_status()
        response_data = ""
        for line in r.iter_lines():
            if not line:
                continue
            response_json = json.loads(line)
            if 'message' in response_json:
                response_data += response_json['message']['content']

    return response_data

In [4]:
result = query_model("hello llama")
print(result)

Hello there! It's not every day I get to chat with a llama! How are you doing today?


In [5]:
def format_input(entry):
    instruction_text = (
        f"Below is an instruction that describes a task. "
        f"Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{entry['instruction']}"
    )

    input_text = (f"\n\n### Input:\n{entry['input']}" if entry["input"] else "")
    return instruction_text + input_text

In [6]:
#lets generate 'chosen' and 'rejected' responses for the prefence finetuning

import random
from tqdm import tqdm

for entry in tqdm(test_data[:3]):
    politeness = random.choice(["polite", "impolite"])
    prompt = (
        f"Given the input `{format_input(entry)}` "
        f"and correct output `{entry['output']}`, "
        f"slightly rewrite the output to be more {politeness}."
        "Keep the modification minimal."
        "Only return the generated response and nothing else(like: here's your response)."
    )

    print(f"\nDataset response:")
    print(">> ", entry['output'])
    print(f"{politeness} response:")
    print(">> ", query_model(prompt))

  0%|          | 0/3 [00:00<?, ?it/s]


Dataset response:
>>  The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".
impolite response:


 33%|███▎      | 1/3 [00:16<00:32, 16.40s/it]

>>  Are you kidding me? The spelling of this pathetic excuse for a word "freind" is completely wrong, it should be spelled out as "friend".

Dataset response:
>>  He goes to the park every day.
impolite response:


 67%|██████▋   | 2/3 [00:26<00:12, 12.90s/it]

>>  He's only going to the park every day, like it's any of our business.

Dataset response:
>>  45 kilometers is 45000 meters.
polite response:


100%|██████████| 3/3 [00:35<00:00, 11.72s/it]

>>  45 kilometers is actually equivalent to 45,000 meters.


In [7]:
#applying on the whole dataset

def generate_model_response(json_data):
    for i,entry in enumerate(tqdm(json_data, desc="writing entries")):
        politeness = random.choice(["polite", "impolite"])
        prompt = (
            f"Given the input `{format_input(entry)}` "
            f"and correct output `{entry['output']}`, "
            f"slightly rewrite the output to be more {politeness}."
            "Keep the modification minimal."
            "Only return the generated response and nothing else(like: here's your response)."
        )

        response = query_model(prompt)
        if politeness == "polite":
            json_data[i]['chosen'] = response
            json_data[i]['rejected'] = entry['output']
        else:
            json_data[i]['chosen'] = entry['output'] 
            json_data[i]['rejected'] = response

In [8]:
generate_model_response(test_data)

writing entries:  26%|██▌       | 283/1100 [45:25<2:11:08,  9.63s/it]


KeyboardInterrupt: 

In [9]:
test_data[0]

{'instruction': 'Evaluate the following phrase by transforming it into the spelling given.',
 'input': 'freind --> friend',
 'output': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".',
 'chosen': 'The spelling of the given phrase "freind" appears to contain an error, with the correct spelling being actually "friend".',
 'rejected': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}